In [ ]:
# Cell 1 [全局模型规模开关 + 挂载 Google Drive 与 GPU 体检]
# ==============================================================================
# 单变量全局开关（供 Cell 5 与 Cell 6 自动读取）：
#   "48m"  -> 47.62M 总参 / 15.77M 激活 (8 experts Top-2, L_max=8)
#   "210m" -> 208.65M 总参 / 22.86M 激活 (32 experts Top-2, L_max=8, 推荐 H100)
# ==============================================================================
MODEL_PRESET = "48m"  # 可切换为 "210m"

import os
import subprocess
from pathlib import Path

if MODEL_PRESET not in ("48m", "210m"):
    raise ValueError(f"MODEL_PRESET must be '48m' or '210m', got {MODEL_PRESET!r}")
os.environ["DCPA_MODEL_PRESET"] = MODEL_PRESET

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
for sub in ("code", "data_archive", "checkpoints", "logs", "export"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(f"[Drive] Ready: {DRIVE_ROOT} | MODEL_PRESET={MODEL_PRESET}")

# GPU health check & 4-tier hardware sharding preview (no hardcoded torch.cuda.get_device_properties)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
)
if smi.returncode == 0 and smi.stdout.strip():
    first_line = smi.stdout.strip().splitlines()[0]
    gpu_name, mem_mib_s = [part.strip() for part in first_line.split(",", 1)]
    vram_gb = float(mem_mib_s) / 1024.0
    if vram_gb >= 75.0:
        tier, micro_bytes, accum = "h100_or_a100_80g", 131072, 1
    elif vram_gb >= 38.0:
        tier, micro_bytes, accum = "a100_40g", 65536, 2
    elif vram_gb >= 20.0:
        tier, micro_bytes, accum = "l4_or_4090", 32768, 4
    else:
        tier, micro_bytes, accum = "cuda_compact", 16384, 8
    print(
        f"[GPU Health] {gpu_name} | VRAM={vram_gb:.1f} GB | tier={tier} | "
        f"micro_batch_bytes={micro_bytes} | grad_accum_steps={accum} | precision=bfloat16"
    )
else:
    print("[GPU Health][WARN] No NVIDIA GPU detected via nvidia-smi; will fall back to CPU float32.")


In [ ]:
# Cell 2 [部署代码至本地 NVMe 与安装依赖]
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

BUNDLE_ZIP = Path("/content/drive/MyDrive/DCPA_Project/code/dcpa_colab_bundle.zip")
CODE_DIR = Path("/content/dcpa")
NVME_CACHE = Path("/content/nvme_cache")

if not BUNDLE_ZIP.exists():
    raise FileNotFoundError(
        f"Missing code bundle at {BUNDLE_ZIP}. Run `bash scripts/pack_for_colab.sh` locally "
        "and upload `dcpa_colab_bundle.zip` to Google Drive first."
    )

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
CODE_DIR.mkdir(parents=True, exist_ok=True)
NVME_CACHE.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(CODE_DIR)

os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{CODE_DIR}:{CODE_DIR / 'dcpa-train-py'}"

deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(CODE_DIR / "dcpa-train-py")],
    check=True,
)

from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print(f"[Deploy] Code unpacked to {CODE_DIR} | HF_HUB_ENABLE_HF_TRANSFER={os.environ['HF_HUB_ENABLE_HF_TRANSFER']}")
print(describe_hardware_profile(profile))


In [ ]:
# Cell 3 [全量 450 MB 底座语料 + 260 MB 三大能力语料智能缓存：首次流式构建归档，重连极速恢复]
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

DRIVE_ARCHIVE = Path("/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_corpus_v1.tar.gz")
DRIVE_CAP_ARCHIVE = Path("/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_cap_v1.tar.gz")
NVME_DATA_DIR = Path("/content/nvme_cache/data")
NVME_CAP_DIR = Path("/content/nvme_cache/cap_data")
NVME_TAR_TMP = Path("/content/nvme_cache/dcpa_corpus_v1.tar.gz")
NVME_CAP_TAR_TMP = Path("/content/nvme_cache/dcpa_cap_v1.tar.gz")

NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
NVME_CAP_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()

# 1. 450 MB 5-domain base pretraining corpus
if DRIVE_ARCHIVE.exists():
    print(f"[Corpus Cache HIT] Restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...")
    shutil.copy2(DRIVE_ARCHIVE, NVME_TAR_TMP)
    with tarfile.open(NVME_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus Cache HIT] Restored in {time.perf_counter() - t0:.2f}s.")
else:
    print("[Corpus Cache MISS] Building full ~450 MB 5-domain corpus directly on local NVMe...")
    subprocess.run([sys.executable, "data/build_corpus.py", "--out-dir", str(NVME_DATA_DIR)], check=True)
    env = dict(os.environ)
    env["DCPA_DATA_DIR"] = str(NVME_DATA_DIR)
    subprocess.run([sys.executable, "-m", "pytest", "-v", "tests/test_data_pipeline.py"], env=env, check=True)
    with tarfile.open(NVME_TAR_TMP, "w:gz") as tf:
        for fname in ("train.bin", "train.idx", "train.meta.json", "val.bin", "val.idx", "val.meta.json", "corpus_meta.json"):
            fpath = NVME_DATA_DIR / fname
            if fpath.exists():
                tf.add(fpath, arcname=fname)
    sync_file_to_drive(NVME_TAR_TMP, DRIVE_ARCHIVE.parent, dest_name=DRIVE_ARCHIVE.name)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus] Archived to {DRIVE_ARCHIVE} in {time.perf_counter() - t0:.1f}s.")

# 2. 260 MB Capability Upgrade Shards (100 MB reasoning + 100 MB writing + 60 MB toolcall)
t1 = time.perf_counter()
if DRIVE_CAP_ARCHIVE.exists():
    print(f"[Cap Cache HIT] Restoring {DRIVE_CAP_ARCHIVE} -> {NVME_CAP_DIR} ...")
    shutil.copy2(DRIVE_CAP_ARCHIVE, NVME_CAP_TAR_TMP)
    with tarfile.open(NVME_CAP_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_CAP_DIR)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    print(f"[Cap Cache HIT] Restored 260 MB capability shards in {time.perf_counter() - t1:.2f}s.")
else:
    print("[Cap Cache MISS] Streaming full 260 MB capability shards (reasoning 100MB + writing 100MB + toolcall 60MB)...")
    subprocess.run([sys.executable, "data/build_cap_reasoning.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100"], check=True)
    subprocess.run([sys.executable, "data/build_cap_writing.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100"], check=True)
    subprocess.run([sys.executable, "data/build_cap_toolcall.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "60"], check=True)
    with tarfile.open(NVME_CAP_TAR_TMP, "w:gz") as tf:
        for fpath in sorted(NVME_CAP_DIR.iterdir()):
            if fpath.is_file():
                tf.add(fpath, arcname=fpath.name)
    sync_file_to_drive(NVME_CAP_TAR_TMP, DRIVE_CAP_ARCHIVE.parent, dest_name=DRIVE_CAP_ARCHIVE.name)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    if hasattr(os, "sync"):
        os.sync()
    print(f"[Cap] Archived 260 MB capability shards to {DRIVE_CAP_ARCHIVE} in {time.perf_counter() - t1:.1f}s.")


In [ ]:
# Cell 4 [云端 CUDA 算子与五大能力升级上线复检]
import subprocess
import sys
import time

t0 = time.perf_counter()
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "tests/test_sanity.py",
        "tests/test_trainer_lifecycle.py",
        "tests/test_capabilities_upgrade.py",
    ],
    check=True,
)
print(f"[Cloud Preflight] Causality, STE gradients, A/B sync & Capability Upgrade 100% GREEN in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 5 [一键启动 / 断连无缝续跑 10 万步底座训练（支持 48M / 210M-MoE + Stage 3 35% 推理写作混训）]
# 注：自动释放显卡逻辑已移至 Cell 6（全部 Stage 1~5 任务完成后统一释放）
import os
import subprocess
import sys
from pathlib import Path

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "48m")
if MODEL_PRESET == "210m":
    CFG_FILE, PRESET_NAME = "configs/dcpa_210m_moe.yaml", "moe_210m"
else:
    CFG_FILE, PRESET_NAME = "configs/dcpa_48m_train.yaml", "proto_48m"

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
NVME_CACHE = Path("/content/nvme_cache")
NVME_DATA_DIR = NVME_CACHE / "data"
NVME_CAP_DIR = NVME_CACHE / "cap_data"
DRIVE_CKPTS = DRIVE_ROOT / "checkpoints"
DRIVE_LOGS = DRIVE_ROOT / "logs"
DRIVE_EXPORT = DRIVE_ROOT / "export"
FINAL_SAFETENSORS = DRIVE_EXPORT / "dcpa_final.safetensors"

cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--config", CFG_FILE,
    "--preset", PRESET_NAME,
    "--total-steps", "100000",
    "--seq-len", "2048",
    "--data-dir", str(NVME_DATA_DIR),
    "--cap-data-dir", str(NVME_CAP_DIR),
    "--local-cache-dir", str(NVME_CACHE),
    "--drive-sync-dir", str(DRIVE_CKPTS),
    "--drive-logs-dir", str(DRIVE_LOGS),
    "--drive-export-dir", str(DRIVE_EXPORT),
    "--save-every-steps", "1000",
    "--save-every-minutes", "10",
    "--probe-every-steps", "500",
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--compile-mode", "auto",
    "--max-consecutive-meltdowns", "3",
]

print(f"[Train][{MODEL_PRESET}] Launching: {' '.join(cmd)}", flush=True)
proc = subprocess.run(cmd, check=False)
exit_code = int(proc.returncode)
if hasattr(os, "sync"):
    os.sync()
if exit_code == 0:
    print(f"[Train] 100,000 steps ({PRESET_NAME}) completed cleanly! Final export: {FINAL_SAFETENSORS}", flush=True)
elif exit_code == 42:
    raise RuntimeError("[Train][CIRCUIT_BREAKER] Aborted after 3 consecutive unrecoverable meltdowns.")
else:
    raise RuntimeError(f"Training exited unexpectedly with code {exit_code}")


In [ ]:
# Cell 6 [Stage 5：8,000 步语用 + 推理/写作/工具全能微调与导出 + 全部完工自动释放显卡]
# ==============================================================================
# AUTO_UNASSIGN_RUNTIME_ON_FINISH:
#   True  -> Stage 5 跑满 8,000 步导出 dcpa_chat_sft.safetensors 至 Google Drive
#            后，强制 os.sync() 等待 10 秒并自动归还 H100/A100 实例，防止空烧 CCU。
#   False -> 保留当前 Colab 实例以便手动调试。
# ==============================================================================
AUTO_UNASSIGN_RUNTIME_ON_FINISH = True

import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
NVME_CACHE = Path("/content/nvme_cache")
NVME_SFT_DIR = NVME_CACHE / "sft_data"
NVME_CAP_DIR = NVME_CACHE / "cap_data"
DRIVE_SFT_ARCHIVE = DRIVE_ROOT / "data_archive" / "dcpa_sft_stage5_v1.tar.gz"
DRIVE_EXPORT = DRIVE_ROOT / "export"
BASE_WEIGHTS = DRIVE_EXPORT / "dcpa_final.safetensors"
CHAT_WEIGHTS = DRIVE_EXPORT / "dcpa_chat_sft.safetensors"

exit_code = -1
try:
    if not BASE_WEIGHTS.exists():
        raise FileNotFoundError(f"Missing Stage 4 base export: {BASE_WEIGHTS}. Run Cell 5 first.")

    # 1. Build or restore ~20 MB 4-bucket Stage 5 SFT dataset on local NVMe
    NVME_SFT_DIR.mkdir(parents=True, exist_ok=True)
    if DRIVE_SFT_ARCHIVE.exists():
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        shutil.copy2(DRIVE_SFT_ARCHIVE, tmp_tar)
        with tarfile.open(tmp_tar, "r:gz") as tf:
            tf.extractall(NVME_SFT_DIR)
        tmp_tar.unlink(missing_ok=True)
        print(f"[Stage 5 Data] Restored cached SFT corpus -> {NVME_SFT_DIR}", flush=True)
    else:
        subprocess.run(
            [sys.executable, "data/build_sft_stage5.py", "--out-dir", str(NVME_SFT_DIR)],
            check=True,
        )
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        with tarfile.open(tmp_tar, "w:gz") as tf:
            for fpath in sorted(NVME_SFT_DIR.iterdir()):
                if fpath.is_file():
                    tf.add(fpath, arcname=fpath.name)
        sync_file_to_drive(tmp_tar, DRIVE_SFT_ARCHIVE.parent, dest_name=DRIVE_SFT_ARCHIVE.name)
        tmp_tar.unlink(missing_ok=True)

    # 2. Run 8,000-step Stage 5 balanced SFT + Reasoning + Writing + Tool-Calling (~20 min on H100)
    sft_cmd = [
        sys.executable,
        "-u",
        "train/train_stage5_sft.py",
        "--weights", str(BASE_WEIGHTS),
        "--data-dir", str(NVME_SFT_DIR),
        "--cap-data-dir", str(NVME_CAP_DIR),
        "--export-dir", str(NVME_CACHE / "export"),
        "--drive-export-dir", str(DRIVE_EXPORT),
        "--total-steps", "8000",
        "--target-tokens-bytes-per-step", "65536",
        "--max-doc-bytes", "2048",
        "--compile-mode", "auto",
    ]
    print(f"[Stage 5 SFT] Launching: {' '.join(sft_cmd)}", flush=True)
    proc = subprocess.run(sft_cmd, check=False)
    exit_code = int(proc.returncode)
    if exit_code == 0 and CHAT_WEIGHTS.exists():
        print(f"[Stage 5 DONE] Chat + Tool SFT weights synced to Drive: {CHAT_WEIGHTS}", flush=True)
    else:
        raise RuntimeError(f"Stage 5 SFT exited with code {exit_code}")
finally:
    if hasattr(os, "sync"):
        os.sync()
    if AUTO_UNASSIGN_RUNTIME_ON_FINISH and exit_code == 0 and CHAT_WEIGHTS.exists():
        print("[Auto-Unassign] Waiting 10s after os.sync() before releasing GPU...", flush=True)
        time.sleep(10)
        from google.colab import runtime

        runtime.unassign()
